# Step 1: Raw Data Exploration & Cropping Preview
This cell generates a quick, uncropped plot of the raw CSV data. The objective is to visually inspect the continuous waveform and identify the exact start and end times (in seconds) of the clean movement cycles you want to analyze. Take note of these timestamps to manually configure the parameters in the next step.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

file_path = 'mama_flexion_codo_normal.csv'
# Shoulder_Elevation_Deg for shoulder or Elbow_Flexion_Deg for elbow
target_axis= 'Elbow_Flexion_Deg'

df = pd.read_csv(file_path)

plt.figure(figsize=(10, 4))
plt.plot(df['Time(s)'], df[target_axis], color='gray')
plt.title(f'Raw Explorer: {file_path}')
plt.xlabel('Time (s)')
plt.ylabel('Degrees')
plt.grid(True, linestyle=':')
plt.show()

print("Observe the graph and decide the exact start and end seconds you want to crop.")

### Step 2: Clinical Kinematic Analysis & Validation
This is the main analysis program. Before running this cell, update the **Analysis Params** variables in the code with the start and end times identified in Step 1, along with the specific clinical parameters for this trial (BPM, baseline resting angle, and target maximum angle).

Based on your inputs, the script will:
1. Apply a zero-phase low-pass Butterworth filter to remove IMU high-frequency noise.
2. Crop the dataset to your specified timeframe.
3. Generate the theoretical ideal wave based on the metronome pace.
4. Calculate key validation metrics (RMSE, Pearson *r* correlation, and Peak Error).
5. Output the final comparative graph.



In [ ]:
# SET UP AND LIBRARIES

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from scipy.stats import pearsonr

plt.style.use('seaborn-v0_8-whitegrid')
print("Libraries loaded successfully. Ready for Data Analysis.")

In [ ]:
# -----------------------------------------
# ANALYSIS PARAMS - ADJUST FOR EACH CSV
# ------------------------------------------

csv_file_path = 'mama_flexion_codo_normal.csv'
target_axis = 'Elbow_Flexion_Deg' # 'Elbow_Flexion_Deg' or 'Shoulder_Elevation_Deg'

# Manual time crop
start_time = 0.0
end_time = 3.5

# Subject Clinical Parameters
metronome_bpm = 60
baseline_resting_angle = 7.5   #Real "0" (what the IMU reads at rest)
target_max_angle = 143.0    #Target degree for that movement

In [ ]:
#FUNCTIONS

#Applies a zero-phase low-pass Butterworth filter to remove high-frequency IMU noise.
def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs # Nyquist Frequency
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

# Generates the theoretical 'Ideal Curve' based on the metronome pace.
def generate_ideal_curve(time_array, target_min, target_max, bpm):
    freq_hz = (bpm / 60) / 2  #1 full cycle (up and down) equals 2 beats
    amplitude = (target_max - target_min) / 2
    offset = target_min + amplitude
    # Generate a sine wave matching the metronome frequency
    ideal_wave = offset - amplitude * np.cos(2 * np.pi * freq_hz * time_array)
    return ideal_wave

In [ ]:
#DATA PROCESSING

df_raw = pd.read_csv(csv_file_path)

#Manual crop
df_cut = df_raw[(df_raw['Time(s)'] >= start_time) & (df_raw['Time(s)'] <= end_time)].copy()

if df_cut.empty:
    print("Error! The crop times are out of the file's bounds.")
else:
    # Reset the timeline so the cropped section starts at 0s
    df_cut['Time_Reset'] = df_cut['Time(s)'] - df_cut['Time(s)'].iloc[0]
    time_array = df_cut['Time_Reset'].values

    # Estimate sampling rate (FPS)
    time_diffs = np.diff(df_cut['Time(s)'])
    fs_estimated = 1.0 / np.mean(time_diffs)

    # Noise filtering
    df_cut['Filtered_Angle'] = butter_lowpass_filter(df_cut[target_axis], cutoff=5.0, fs=fs_estimated)

    # Generate ideal curve
    ideal_curve = generate_ideal_curve(time_array, baseline_resting_angle, target_max_angle, metronome_bpm)
    unity_curve = df_cut['Filtered_Angle'].values


In [ ]:
# METRICS CALCULATION

# RMSE (Root Mean Square Error)
rmse = np.sqrt(np.mean((unity_curve - ideal_curve)**2))

# Peak Error (Maximum absolute difference)
peak_error = np.max(np.abs(unity_curve - ideal_curve))

# Pearson Correlation (r)
pearson_r, _ = pearsonr(unity_curve, ideal_curve)

print(f"--- METRICS FOR {file_path} ---")
print(f"RMSE:          {rmse:.2f}º")
print(f"Peak Error:    {peak_error:.2f}º")
print(f"Pearson (r):   {pearson_r:.4f}")


In [ ]:
# VISUALIZATION

plt.figure(figsize=(12, 6))

# Plot the curves
plt.plot(df_trimmed['Time(s)'], df_trimmed[target_axis], color='lightgray', label='Raw Data', alpha=0.6)
plt.plot(df_trimmed['Time(s)'], unity_curve, color='#1f77b4', label='Filtered signal (IMU)', linewidth=2.5)
plt.plot(df_trimmed['Time(s)'], ideal_curve, color='#d62728', linestyle='--', label='Ideal Target (Metronome)', linewidth=2)


plt.title(f'Kinematic Validation: {file_path.replace(".csv", "")}', pad=15, fontweight='bold')
plt.xlabel('Time (seconds)', fontweight='bold')
plt.ylabel('Angle (degrees)', fontweight='bold')
plt.legend(loc='upper right', frameon=True, shadow=True)
plt.grid(True, linestyle=':', alpha=0.7)

# Save the plot
output_image_name = csv_file_path.replace('.csv', '_plot.png')
plt.savefig(output_image_name, dpi=300, bbox_inches='tight')
print(f"Plot saved successfully as: {output_image_name}")

plt.show()